# Monday: words, embeddings and LLM labels

Three ways to turn many texts into something you can count: word weights (TF-IDF), embeddings, and labels from an LLM. You try each one on comments from r/AmItheAsshole, where the commenter's verdict tells you the answer. Then you label posts for something no single word marks: does the poster admit they did something wrong?

Work with a partner. Write code in the cells that start with a comment, and use the check comments to confirm your result. Make a copy named `week10_live_workspace_mine.ipynb`. Part 5 calls Gemini, so you need `GEMINI_API_KEY` in your `.env` file. It makes 9 calls, which cost less than one cent.

In [ ]:
import warnings
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore", category=FutureWarning)  # a harmless warning from one of the libraries
pd.set_option("display.max_colwidth", 300)  # show more of each text

## 1. Comments with a known verdict

Before you label anything, check whether you already have the label. Posts on r/AmItheAsshole carry the community's verdict as flair, so there is nothing to label there. This comment file has only an id and the text, with no verdict column.

But most comments start with a verdict: NTA (not the asshole) or YTA (you're the asshole). That first word is a label the commenter gave, so you know the answer for each of these comments. The cell keeps the comments that start with NTA or YTA and removes the verdict word from the text. Later you check other methods against that word.

In [ ]:
comments = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/docs/interactives/data/week08/aita_replies.csv").rename(columns={"idstr": "id"})
comments["verdict"] = comments["body"].str.upper().str.extract(r"^(NTA|YTA)\b")[0]
comments = comments.dropna(subset=["verdict"]).reset_index(drop=True)
comments["text"] = (comments["body"]
                    .str.replace(r"\b(?:NTA|YTA|ESH|NAH|INFO)\b", "", regex=True, case=False)
                    .str.split().str.join(" ").str.lstrip(" .,:;!-–—"))
comments[["verdict", "text"]].head()  # Check: 1035 rows in comments

In [ ]:
# The share of comments with each verdict. Check: NTA 0.63, YTA 0.37

## 2. Words: which words go with each verdict?

TF-IDF gives each comment a weight for each word. The average weight in YTA comments and in NTA comments shows which words go with each verdict. You need the groups first: TF-IDF describes them. It does not label a comment.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", min_df=15)  # words in at least 15 comments
weights = pd.DataFrame(tfidf.fit_transform(comments["text"]).toarray(), columns=tfidf.get_feature_names_out())
by_verdict = weights.groupby(comments["verdict"]).mean().T  # one row per word: its average weight in NTA and in YTA comments
by_verdict["ratio"] = by_verdict["YTA"] / by_verdict["NTA"]
by_verdict.shape  # Check: (220, 3)

In [ ]:
# Show the 10 words with the highest ratio (YTA) and the 10 with the lowest (NTA).
# Check: the highest are fun, look, knew, world, dude; the lowest are behavior, respect, able, change, won

What kind of comment do the YTA words suggest? What can't these weights tell you about one comment?

Your answer:

## 3. Embeddings: find comments about a topic

An embedding turns each comment into 384 numbers, so that comments about similar things get similar numbers. Say your project asks how often commenters warn a poster about a controlling partner. A word search finds only the comments that use the word. A search with a sentence also finds comments that say the same thing in other words.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # the first run downloads the model, about 90 MB
embeddings = model.encode(comments["text"].tolist(), normalize_embeddings=True)
embeddings.shape  # Check: (1035, 384)

In [ ]:
def search(sentence, n=5):
    """The n comments closest to a sentence."""
    query = model.encode([sentence], normalize_embeddings=True)[0]
    found = comments.assign(similarity=embeddings @ query)
    return found.sort_values("similarity", ascending=False).head(n)[["similarity", "verdict", "text"]]

search("Your partner is controlling you.")  # Check: the second comment starts "your girlfriend is disregarding"

In [ ]:
# How many comments use the word "controlling"? Use str.contains with case=False.
# Check: 6. Which of the five comments above would a word search miss?

In [ ]:
# Search with a sentence about a topic from your project. Then count the comments that contain its main word.
# Check: the search finds comments without that word

Do the comments the search finds agree on whether the poster was in the wrong? What does that tell you about using embeddings to measure a judgment?

Your answer:

## 4. An LLM label: does the comment say the poster was in the wrong?

A label answers a question you define, one text at a time. Because the verdict word was removed, you can check the label against it. The file below has 20 comments, 10 NTA and 10 YTA, labeled by five methods:

- `word_search`: YTA if the comment contains words such as *rude*, *apologize* or *should have*, otherwise NTA
- `classifier`: TF-IDF and logistic regression, trained on other comments
- `gemini_short`: Gemini with the prompt `Label this reply NTA or YTA.`
- `gemini_careful`: Gemini with a system instruction that says the verdict was removed and defines NTA and YTA
- `gemini_tone`: Gemini with the prompt `Is this comment positive or negative?`, where negative counts as YTA

In [ ]:
reply_labels = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_reply_labels.csv")
METHODS = ["word_search", "classifier", "gemini_short", "gemini_careful", "gemini_tone"]
reply_labels[["verdict", "text"] + METHODS].head()  # Check: 20 rows in reply_labels

In [ ]:
# Print the accuracy of each method in METHODS.
# Check: word_search 0.55, classifier 0.35, gemini_short 0.8, gemini_careful 0.8, gemini_tone 0.6

Say you are looking for YTA comments. A method can make two kinds of mistakes:

- A **false positive** is an NTA comment labeled YTA: the method finds something that is not there.
- A **false negative** is a YTA comment labeled NTA: the method misses something that is there.

Accuracy adds the two together. In a confusion table, they are the two cells off the diagonal.

*Hint: the false positives of method `m` are the rows where `(reply_labels[m] == "YTA") & (reply_labels["verdict"] == "NTA")`.*

In [ ]:
pd.crosstab(reply_labels["verdict"], reply_labels["gemini_tone"])  # rows: true verdict, columns: label

In [ ]:
# For each method, count the false positives and the false negatives.
# Check: gemini_short 0 and 4, gemini_careful 2 and 2, gemini_tone 7 and 1

In [ ]:
# The share of comments each method labels YTA. The true share is 0.5.
(reply_labels[METHODS] == "YTA").mean()

In [ ]:
# The NTA comments the tone prompt labeled YTA, with Gemini's reason
reply_labels.loc[(reply_labels["verdict"] == "NTA") & (reply_labels["gemini_tone"] == "YTA"), ["text", "gemini_tone_reason"]]

`gemini_short` and `gemini_careful` have the same accuracy. Which kind of mistake does each make? Which would you use to estimate the share of YTA comments, and which to collect YTA comments to read? The classifier gets the share almost right with an accuracy of 0.35. How?

Your answer:

## 5. A harder label: does the poster admit fault?

No single word in a post says whether the poster admits fault, and the community's verdict is a different thing. Gemini labeled 40 short posts, 10 of each verdict, with the instruction below. To check it, you need your own labels.

In [ ]:
FIRST_SYSTEM = """You read posts from the Reddit forum r/AmItheAsshole, where people describe a conflict and ask whether they were in the wrong.

Label whether the poster admits that they did something wrong.
- yes: the poster says that something they did was wrong, unfair, or an overreaction, even in passing.
- no: the poster does not admit any fault.

Give a reason of one sentence."""

In [ ]:
fault = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_admits_fault.csv")
post_texts = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/aita_top_subs.csv")[["idstr", "title", "selftext"]]
fault = fault.merge(post_texts, on="idstr")
fault[["verdict", "words", "llm_label", "title"]].head()  # Check: 40 rows in fault

In [ ]:
# The 8 shortest posts. Read each one.
mine = fault.sort_values("words").head(8).reset_index(drop=True)
for i, post in mine.iterrows():
    print(i, post["title"], "\n", post["selftext"], "\n")

In [ ]:
# Your label for each of the 8 posts, in order: "yes" or "no"
my_labels = ["...", "...", "...", "...", "...", "...", "...", "..."]

Compare your labels with the pair next to you. Where you disagree, is the definition unclear, or the post?

In [ ]:
# Add my_labels to mine as a column, and compute how often Gemini's label (llm_label) agrees with yours.
# Check: a number between 0 and 1

### Change the instruction

Change one thing in the definition so that Gemini's labels come closer to yours, and run it on the same 8 posts.

In [ ]:
# Supplied: connect to Gemini. The key is read from your .env file and never printed.
import json
import logging
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env", override=True)
        break
key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"
logging.getLogger("google_genai.types").setLevel(logging.ERROR)  # a warning google-genai prints on every call
print("Connected to Gemini.")

In [ ]:
# Supplied: the schema, and a function that sends one post with a system instruction
SCHEMA = {
    "type": "object",
    "properties": {
        "llm_label": {"type": "string", "enum": ["yes", "no"]},
        "llm_reason": {"type": "string"},
    },
    "required": ["llm_label", "llm_reason"],
}


def build_prompt(title, text):
    return f"Title: {title}\n\n{text}"


def ask(prompt, system, max_tries=4):
    """Send one prompt with this system instruction. Return the label and the reason as a dictionary."""
    config = types.GenerateContentConfig(system_instruction=system, response_mime_type="application/json", response_schema=SCHEMA)
    wait = 10
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=config)
            time.sleep(6)  # stay under the free limit of calls per minute
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in {429, 503} or attempt == max_tries:
                raise
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")  # 429: too many calls, 503: server busy
            time.sleep(wait)
            wait = wait * 2

In [ ]:
# Ask Gemini about the first post in mine with FIRST_SYSTEM, and print the answer.
# Use ask(build_prompt(title, text), system). Check: a dictionary with llm_label and llm_reason

In [ ]:
# Copy FIRST_SYSTEM here and change one thing in the definition.
MY_SYSTEM = """..."""

In [ ]:
# Label the 8 posts with your instruction: 8 calls, about a minute
assert "..." not in MY_SYSTEM, "Write your instruction in MY_SYSTEM first."
new_labels = []
for post in mine.itertuples():
    answer = ask(build_prompt(post.title, post.selftext), MY_SYSTEM)
    new_labels.append(answer["llm_label"])
    print(len(new_labels), "of", len(mine), "done")
mine["new_label"] = new_labels

In [ ]:
# How often does each instruction agree with your labels: FIRST_SYSTEM (llm_label) and yours (new_label)?

Did your change bring Gemini closer to your labels? On 8 posts, one post is 12.5 percentage points. Would you trust the difference?

Your answer:

## 6. Use the labels

Labels let you answer a question about many posts at once: do posters who admit fault get YTA more often?

In [ ]:
# The share of posts Gemini labeled yes, for each verdict. Check: ESH 0.1, NAH 0.2, NTA 0.0, YTA 0.1

What would you need before you report this result? Think of your check in Part 5 and of how many posts the shares rest on.

Your answer:

## 7. Your project: words, embeddings or labels?

| | Words (TF-IDF) | Embeddings | LLM labels |
|---|---|---|---|
| Question | Which words go with a group? | Which texts are about the same thing? | Does each text have the property you define? |
| You need | Groups you already have | Nothing else | A definition, and texts you labeled by hand to check |
| It misses | Meaning beyond single words | The stance: what a text says about its topic | Nothing is free: it costs money and can be wrong |

Answer these for your project, with the 50 texts you label in HW4 in mind.

1. Your research question:
2. Which of the three answers it, and why:
3. If labels: the concept, your labels with one sentence each, plus `unclear`:
4. Could a keyword rule find the labels? Name one word you would try.
5. Which of your 50 texts would you use to write a prompt, and which to check it?

Your answers:

## If you have time

### Precision and recall

Two common summaries of the mistakes, for the label you are looking for (here YTA):

- **Precision**: of the comments labeled YTA, the share that are YTA.
- **Recall**: of the YTA comments, the share labeled YTA.

Compute both for each method in Part 4. Which method has the highest precision, and which the highest recall?

In [ ]:
# Precision and recall for YTA. Check: gemini_short 1.0 and 0.6, gemini_tone 0.56 and 0.9